This tutorial and the assets can be downloaded as part of the [Wallaroo Tutorials repository](https://github.com/WallarooLabs/Wallaroo_Tutorials/blob/wallaroo2025.2_tutorials/wallaroo-model-operations-tutorials/deploy/sidekick-logs).

## Pipeline Sidekick Logs

Pipelines with models deployed in the **Wallaroo Containerized Runtime** store log outputs of their operations as **sidekick  pod logs**.  These represent the outputs of Kubernetes or Podman pods, Python script outputs of operations for Python and Wallaroo Custom Models, etc.

These logs are useful for determining where errors may have occurred during deployment or inference steps.  For issues involving errors during model upload, see [Model Auto-Packaging Troubleshooting]({{<ref "wallaroo-deployment-model-troubleshooting">}}).

Models deployed in the [Wallaroo Containerized Runtime]({{<ref "wallaroo-deployment-upload">}}) store the outputs of the Kubernetes or Podman pod logs, referred to as **pipeline sidekick pod logs**.  These logs have the following qualities:

* Each pipeline sidekick pod log is a specific output for the deployed model in its replica.  For example, if the pipeline deployment configuration uses multiple replicas, the same model is deployed multiple times, each with its own pod logs.
* Pipeline sidekick pod logs are tied to the specific model version deployed in the pipeline.  For example, if the pipeline has the model step named `sample-model`, and the pipeline is deployed first with `sample-model` version A, and later `sample-model` version B, each deployed model version has its own sidekick pod logs.
* Logs are deleted from oldest to newest with data retention clean up run regularly enough to prevent reaching the DB storage limits.
* Returned log results limit is 1 million log lines or 10 MB, whichever limit is reached first.  For large log amounts, restrict the date and time requests to collect smaller amounts.

## Tutorial Goals

The following tutorial is a demonstration of various methods of retrieving sidekick pod logs and error messages from improper requests.

## Tutorial Steps

### Import Libraries

The first step is to import the Python libraries used for this tutorial, including the Wallaroo SDK.

In [1]:
import time
import wallaroo
import requests
import json
import base64
import os
import pyarrow as pa
from wallaroo.framework import Framework
from wallaroo.deployment_config import DeploymentConfigBuilder

wl = wallaroo.Client()

Please log into the following URL in a web browser:

	https://mitch4.wallaroocommunity.ninja/auth/realms/master/device?user_code=AMPW-RPAU

Login successful!


In [33]:
workspace_name = wl.get_current_workspace().name() # "ci@x.com - Default Workspace"
workspace_id = wl.get_current_workspace().id()
pipeline_name = "double-noop-x"
url = f"{wl.api_endpoint}/v1/api/pipelines/get_sidekick_pod_logs"

In [25]:
pipeline = wl.list_pipelines()[3]

In [26]:
sk_name=pipeline.status()['sidekicks'][0]['name']

In [ ]:
model_1 = wl.search_models("noop-py-pre")[0]
model_2 = wl.search_models("noop-onnx")[0]
model_3 = wl.search_models("noop-py-post")[0]

In [ ]:
# pipeline with a couple of sidekicks

input_schema, output_schema = (
    pa.schema([pa.field("tensor", pa.list_(pa.float32()))]),
    pa.schema([pa.field("tensor", pa.list_(pa.float32()))]),
)

model_1 = wl.upload_model(
    "noop-py-pre",
    "passthrough.zip",
    framework=Framework.PYTHON,
    input_schema=input_schema,
    output_schema=output_schema,
)

model_2 = wl.upload_model(
    "noop-onnx",
    "no-op-floats.onnx",
    framework=Framework.ONNX,
).configure(tensor_fields=["tensor"])

input_schema, output_schema = (
    pa.schema([pa.field("outputs", pa.list_(pa.float32()))]),
    pa.schema([pa.field("output", pa.list_(pa.float32()))]),
)

model_3 = wl.upload_model(
    "noop-py-post",
    "passthrough_2.zip",
    framework=Framework.PYTHON,
    input_schema=input_schema,
    output_schema=output_schema,
)

pipeline = (
    wl.build_pipeline(pipeline_name)
    .add_model_step(model_1)
    .add_model_step(model_2)
    .add_model_step(model_3)
)

In [ ]:
deployment_config = (
    DeploymentConfigBuilder().cpus(0.5).memory("100Mi").replica_count(1)
    .sidekick_cpus(model_1, 0.1)
    .sidekick_cpus(model_3, 0.1)
    .sidekick_memory(model_1, "128Mi")
    .sidekick_memory(model_3, "128Mi")
    .build()
)
pipeline.deploy(deployment_config=deployment_config)

In [35]:
# Normal case

headers = wl.auth.auth_header()
data = { "sidekick_name": sk_name,   # should match sidekicks in pipeline.status
        "workspace_name": workspace_name,
        "pipeline_name": pipeline_name,
         "limit": 1000
       }
response = requests.post(url, headers=headers, json=data)
response.json()

{'logs': ['2026-03-31T20:26:59.883566102Z stdout F \x1b2026-03-31T20:26:59.883463Z\x1b \x1b INFO\x1b \x1buseenv\x1b\x1b:\x1b Starting model deployment',
  '2026-03-31T20:26:59.883611137Z stdout F \x1b2026-03-31T20:26:59.883525Z\x1b \x1b INFO\x1b \x1buseenv\x1b\x1b:\x1b \x1bconfig\x1b\x1b=\x1bConfig { minio_url: "http://minio.wallaroo.svc.cluster.local:9000", minio_base: "/model-bucket", minio_user: "minio", minio_pass: "dSE1NUWyhqxQmUhjFUrLVD8FGRDdLFqu", model_sha: "167b50597f55e1fc774aefb7622d75f03ef542c6479971b8eb2de6210ab01813", venv_path: "/venvs/167b50597f55e1fc774aefb7622d75f03ef542c6479971b8eb2de6210ab01813", nats_host: "nats.wallaroo.svc.cluster.local", pipeline_config_subject: "stream.3.depman.deployment.1.deploy", install_qaic_vllm: false, install_sglang_rocm: false, install_vllm: false, skip_downloads: false, prestaged_venv_tar: None, prestaged_model: None, prestaged_qpcs: None }',
  '2026-03-31T20:26:59.883623988Z stdout F \x1b2026-03-31T20:26:59.883540Z\x1b \x1b INFO\x1b \

In [34]:
# missing "engine-sidekick-" prefix case -- same as model name
# We might want a specific message here to be instructive but I left it the same as all the others.

headers = wl.auth.auth_header()
data = { "sidekick_name": "noop-py-pre",
        "workspace_name": workspace_name,
        "pipeline_name": pipeline_name,
         "limit": 9
       }
response = requests.post(url, headers=headers, json=data)
response.json()

{'msg': "Sidekick 'noop-py-pre' not found in pipeline", 'code': 400}

In [36]:
# workspace not found

headers = wl.auth.auth_header()
data = { "sidekick_name": "engine-sidekick-noop-py-post",
        "workspace_name": "XXXXX",
        "pipeline_name": pipeline_name,
         "limit": 4
       }
response = requests.post(url, headers=headers, json=data)
response.json()

{'msg': 'Workspace name not found: XXXXX', 'code': 400}

In [37]:
# pipeline not found

headers = wl.auth.auth_header()
data = { "sidekick_name": "engine-sidekick-noop-py-post",
        "workspace_name": workspace_name,
        "pipeline_name": "YYYYYYYYY",
         "limit": 4
       }
response = requests.post(url, headers=headers, json=data)
response.json()

{'msg': 'Pipeline not found: YYYYYYYYY', 'code': 400}

In [39]:
# Good dates

headers = wl.auth.auth_header()
data = { "sidekick_name": sk_name,
        "workspace_name": workspace_name,
        "pipeline_name": pipeline_name,
        "limit": 4,
        "start_datetime": "2016-02-16T22:06:03.225401285Z",
        "end_datetime": "2016-02-16T22:08:03.225401285Z",
       }
response = requests.post(url, headers=headers, json=data)
response.json()

{'logs': []}

In [40]:
# Bad date

headers = wl.auth.auth_header()
data = { "sidekick_name": sk_name,
        "workspace_name": workspace_name,
        "pipeline_name": pipeline_name,
        "limit": 4,
        "start_datetime": "xx2026-02-16T22:06:03.225401285Z",
        "end_datetime": "2026-02-16T22:08:03.225401285Z",
       }
response = requests.post(url, headers=headers, json=data)
response.text

'{"msg":"Invalid datetime format: xx2026-02-16T22:06:03.225401285Z. Expected RFC3339 with optional fractional seconds (2024-01-01T10:00:00Z, 2024-01-01T10:00:00.123Z, 2024-01-01T10:00:00.123456789Z), ISO (2024-01-01T10:00:00), or date-only (2024-01-01)","code":400}'

# ... Reference ...

In [35]:
# In case we want to use /status/get_deployment. Notice the compound name argument.

# SDK does this to deployment {"name": f"{self.name()}-{self.id()}"}
# (Pdb) pipeline._deployment.name()
# 'noop-pipeline-748292'
# (Pdb) pipeline._deployment.id()
# 5
# (Pdb) pipeline._deployment.id()
deployment_name = f"{pipeline._deployment.name()}-{pipeline._deployment.id()}"
deployment_name

'double-noop-20'

In [36]:
headers = wl.auth.auth_header()
url = f"{wl.api_endpoint}/v1/api/status/get_deployment"

data = { "name": deployment_name}
response = requests.post(url, headers=headers, json=data)
response.json()

{'status': 'Running',
 'details': [],
 'engines': [{'ip': '10.212.1.89',
   'name': 'engine-0',
   'status': 'Running',
   'reason': None,
   'details': [],
   'pipeline_statuses': {'pipelines': [{'id': 'double-noop',
      'status': 'Running',
      'version': '8a9903c8-80d6-4faa-baef-edd2b29ad9a0'}]},
   'model_statuses': {'models': [{'model_version_id': 174,
      'name': 'noop-py-pre',
      'sha': 'aa1caff27adb57fd246ad3ba9b98706f114953d8c39a71c750d639881dfecb13',
      'status': 'Running',
      'version': 'b59b3dab-5422-4c0c-b0b2-f87ce6c9919d'},
     {'model_version_id': 175,
      'name': 'noop-onnx',
      'sha': '4dc88d159249ccce83942ada69b919cb91455d5fd0e4bfc287de3f21d1aafb1b',
      'status': 'Running',
      'version': '7328ac10-886b-44b4-aa1a-24f416d51ccf'},
     {'model_version_id': 176,
      'name': 'noop-py-post',
      'sha': '167b50597f55e1fc774aefb7622d75f03ef542c6479971b8eb2de6210ab01813',
      'status': 'Running',
      'version': '578cf1d9-3589-44ca-ac60-7afdac

In [21]:
# You can also get to model names via pipeline get_version

headers = wl.auth.auth_header()
url = f"{wl.api_endpoint}/v1/api/pipelines/get_version"

data = { "version": "8a9903c8-80d6-4faa-baef-edd2b29ad9a0"}
response = requests.post(url, headers=headers, json=data)
response.json()

{'pipeline_id': 'double-noop',
 'version': '8a9903c8-80d6-4faa-baef-edd2b29ad9a0',
 'definition': {'id': 'double-noop',
  'steps': [{'ModelInference': {'models': [{'name': 'noop-py-pre',
       'sha': 'aa1caff27adb57fd246ad3ba9b98706f114953d8c39a71c750d639881dfecb13',
       'version': 'b59b3dab-5422-4c0c-b0b2-f87ce6c9919d'}]}},
   {'ModelInference': {'models': [{'name': 'noop-onnx',
       'sha': '4dc88d159249ccce83942ada69b919cb91455d5fd0e4bfc287de3f21d1aafb1b',
       'version': '7328ac10-886b-44b4-aa1a-24f416d51ccf'}]}},
   {'ModelInference': {'models': [{'name': 'noop-py-post',
       'sha': '167b50597f55e1fc774aefb7622d75f03ef542c6479971b8eb2de6210ab01813',
       'version': '578cf1d9-3589-44ca-ac60-7afdaca32c47'}]}}]}}

In [87]:
# how to obtain all sidekick full names
[sk['name'] for sk in pipeline.status()['sidekicks']]

'engine-sidekick-noop-py-post-176-0'

In [44]:
# history API 

headers = wl.auth.auth_header()
data = { "pipeline_name": "double-noop-x",  "workspace_id": workspace_id}
response = requests.post(f"{wl.api_endpoint}/v1/api/pipelines/get_sidekick_pods_history", headers=headers, json=data)
response.json()

{'pods': [{'sidekick_name': 'engine-sidekick-noop-py-pre-b-5-0',
   'started_at': '2026-03-31T20:30:51.797Z',
   'pipeline_model_step': 'noop-py-pre-b',
   'model_sha': '9b634f014101dc88c2a31ba9018dc5d0f364f5e44f4d4923f49078c9bd56cbed',
   'model_version': '1d9f841a-5a2e-496c-98e3-603b57b050fc'},
  {'sidekick_name': 'engine-sidekick-noop-py-post-7-0',
   'started_at': '2026-03-31T20:27:51.798Z',
   'pipeline_model_step': 'noop-py-post',
   'model_sha': '167b50597f55e1fc774aefb7622d75f03ef542c6479971b8eb2de6210ab01813',
   'model_version': '4f3b5c97-566c-4ae0-8f7f-aa3697c5d530'}]}

## Get sidekick logs from SDK

In [17]:
import datetime
start_datetime = datetime.datetime.now() - datetime.timedelta(days=365)
end_datetime = datetime.datetime.now()
# end_datetime = start_datetime

In [41]:
pipeline.get_sidekick_pod_logs(sidekick_name=sk_name, start_datetime=start_datetime, end_datetime=end_datetime, limit=1000000)

2026-03-31T20:26:59.883566102Z stdout F 2026-03-31T20:26:59.883463Z  INFO useenv: Starting model deployment
2026-03-31T20:26:59.883611137Z stdout F 2026-03-31T20:26:59.883525Z  INFO useenv: config=Config { minio_url: "http://minio.wallaroo.svc.cluster.local:9000", minio_base: "/model-bucket", minio_user: "minio", minio_pass: "dSE1NUWyhqxQmUhjFUrLVD8FGRDdLFqu", model_sha: "167b50597f55e1fc774aefb7622d75f03ef542c6479971b8eb2de6210ab01813", venv_path: "/venvs/167b50597f55e1fc774aefb7622d75f03ef542c6479971b8eb2de6210ab01813", nats_host: "nats.wallaroo.svc.cluster.local", pipeline_config_subject: "stream.3.depman.deployment.1.deploy", install_qaic_vllm: false, install_sglang_rocm: false, install_vllm: false, skip_downloads: false, prestaged_venv_tar: None, prestaged_model: None, prestaged_qpcs: None }
2026-03-31T20:26:59.883623988Z stdout F 2026-03-31T20:26:59.883540Z  INFO useenv: Fetching pipeline deployment event from NATS
2026-03-31T20:26:59.883627443Z stdout F 2026-03-31T20:26:59.883566Z  INFO useenv::nats_client: Fetching pipeline deploy event subject=stream.3.depman.deployment.1.deploy skip_downloads=false
2026-03-31T20:26:59.901215303Z stdout F 2026-03-31T20:26:59.900965Z  INFO async_nats: event: connected
2026-03-31T20:26:59.904556108Z stdout F 2026-03-31T20:26:59.904379Z  INFO useenv: Extracted GPU count gpu_count=0
2026-03-31T20:26:59.904581085Z stdout F 2026-03-31T20:26:59.904401Z  INFO useenv: Model configuration extracted model_name=passthrough_2.zip framework=Python openai_enabled=false runtime=Flight
2026-03-31T20:26:59.904583328Z stdout F 2026-03-31T20:26:59.904431Z  INFO storage::s3compat: Memory limit set to 1073741824
2026-03-31T20:26:59.904585701Z stdout F 2026-03-31T20:26:59.904443Z  INFO useenv: Downloading virtual environment
2026-03-31T20:26:59.904587371Z stdout F 2026-03-31T20:26:59.904448Z  INFO useenv::storage_client: Downloading and extracting tar from S3 s3_path=167b50597f55e1fc774aefb7622d75f03ef542c6479971b8eb2de6210ab01813.venv.tar extract_to=/
2026-03-31T20:26:59.904600045Z stdout F 2026-03-31T20:26:59.904515Z  INFO useenv::storage_client: Streaming download to temp file temp_path=/tmp/useenv-29e0249b-b31a-4815-84a2-788b2aef922a.tar
2026-03-31T20:27:38.44930103Z stdout F 2026-03-31T20:27:38.449061Z  INFO useenv: Merging model configurations
2026-03-31T20:27:38.449343329Z stdout F 2026-03-31T20:27:38.449090Z  INFO useenv::model: Merging model configurations model_json=/venvs/167b50597f55e1fc774aefb7622d75f03ef542c6479971b8eb2de6210ab01813/model.json
2026-03-31T20:27:38.449647158Z stdout F 2026-03-31T20:27:38.449489Z  INFO useenv: Merged model configuration:
2026-03-31T20:27:38.449676677Z stdout F {
2026-03-31T20:27:38.449683366Z stdout F   "data": {
2026-03-31T20:27:38.449689757Z stdout F     "id": "827c7906-4579-424e-850b-1a4deb323758",
2026-03-31T20:27:38.449692432Z stdout F     "model": {
2026-03-31T20:27:38.449695069Z stdout F       "config": {
2026-03-31T20:27:38.449698274Z stdout F         "batch_config": null,
2026-03-31T20:27:38.449702038Z stdout F         "continuous_batching_config": null,
2026-03-31T20:27:38.449704719Z stdout F         "dynamic_batching_config": null,
2026-03-31T20:27:38.449707161Z stdout F         "filter_threshold": null,
2026-03-31T20:27:38.449709322Z stdout F         "id": 10,
2026-03-31T20:27:38.449713266Z stdout F         "input_schema": "/////6gAAAAQAAAAAAAKAAwABgAFAAgACgAAAAABBAAMAAAACAAIAAAABAAIAAAABAAAAAEAAAAEAAAA0P///wAAAQwUAAAAIAAAAAQAAAABAAAAKAAAAAcAAABvdXRwdXRzAAQABAAEAAAAEAAUAAgABgAHAAwAAAAQABAAAAAAAAEDEAAAABwAAAAEAAAAAAAAAAQAAABpdGVtAAAGAAgABgAGAAAAAAABAAAAAAA=",
2026-03-31T20:27:38.449725298Z stdout F         "model_version_id": 7,
2026-03-31T20:27:38.449727187Z stdout F         "openai": null,
2026-03-31T20:27:38.449728821Z stdout F         "output_schema": "/////6gAAAAQAAAAAAAKAAwABgAFAAgACgAAAAABBAAMAAAACAAIAAAABAAIAAAABAAAAAEAAAAEAAAA0P///wAAAQwUAAAAIAAAAAQAAAABAAAAKAAAAAYAAABvdXRwdXQAAAQABAAEAAAAEAAUAAgABgAHAAwAAAAQABAAAAAAAAEDEAAAABwAAAAEAAAAAAAAAAQ

In [42]:
pipeline.get_sidekick_pods_history()

sidekick_name,started_at,pipeline_model_step,model_sha,model_version
engine-sidekick-noop-py-pre-b-5-0,2026-03-31T20:30:51.797Z,noop-py-pre-b,9b634f014101dc88c2a31ba9018dc5d0f364f5e44f4d4923f49078c9bd56cbed,1d9f841a-5a2e-496c-98e3-603b57b050fc
engine-sidekick-noop-py-post-7-0,2026-03-31T20:27:51.798Z,noop-py-post,167b50597f55e1fc774aefb7622d75f03ef542c6479971b8eb2de6210ab01813,4f3b5c97-566c-4ae0-8f7f-aa3697c5d530


In [ ]:
import datetime
start_today_datetime = datetime.datetime.now() - datetime.timedelta(hours=1)
end_datetime = datetime.datetime.now()

In [ ]:
pipeline.sidekick_pod_logs(sidekick_name="engine-sidekick-noop-py-post", start_datetime=start_today_datetime, end_datetime=end_datetime, limit=4)

# Private internals

In [45]:

headers = wl.auth.auth_header()
data = { "pipeline_name": "double-noop-x",  "workspace_id": workspace_id}
response = requests.post(f"{wl.api_endpoint}/v1/api/pipelines/get_pipeline_by_name", headers=headers, json=data)
response.json()

{'pipeline': {'id': 10,
  'name': 'double-noop-x',
  'created_at': '2026-03-31T18:51:34.479225+00:00',
  'updated_at': '2026-03-31T18:55:05.962517+00:00'},
 'workspace': {'id': 3,
  'name': '',
  'created_by': None,
  'archived': False,
  'created_at': '2026-04-01T18:10:25.701424206+00:00',
  'group_id': None},
 'plateau_topic': 'workspace-3-pipeline-double-noop-x-inference'}